# Notebook 3 — CNN de referencia

Entrena la red convolucional contra la que se compara el scattering, y cuenta
parámetros de forma explícita.

## El protocolo, que es lo que está en juego

La tesis del trabajo es que el scattering aventaja a una CNN cuando hay pocos
datos. Esa afirmación no vale nada si la CNN está mal entrenada, así que el
protocolo se fija de antemano y no se toca:

1. **La CNN nunca ve más de $n$ muestras.** El split de validación para la
   parada temprana sale de dentro de esas $n$.
2. **Reentrenamiento con las $n$ completas.** Elegida la época óptima sobre
   validación, se reentrena desde cero con todas las muestras durante ese
   número de épocas. Sin este paso la CNN entrenaría con $0.8n$ mientras la SVM
   y el PCA, vía el `refit` de `GridSearchCV`, usan las $n$ completas: el sesgo
   iría en contra de la CNN y a favor de nuestra hipótesis.
3. **Época elegida sobre la curva suavizada.** Con $n=300$ el conjunto de
   validación tiene 60 muestras y su error salta de 1.67 en 1.67 puntos; un
   `argmin` crudo elegiría ruido.
4. **El mismo protocolo a cada $n$**, sin excepciones.

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.cnn import SmallCNN, count_parameters, train_and_evaluate
from src.data import load_mnist, pad_to_square, stratified_subsample, train_val_split
from src.evaluation import Result, save_results
from src.plotting import plot_validation_curves, save_figure, use_paper_style
from src.repro import environment_report, set_seed

SIZE, SEED = 32, 0
TRAIN_SIZES = [300, 1000, 5000]
VAL_FRACTION = 0.2

set_seed(SEED)
use_paper_style()
environment_report().to_dict()

## Arquitectura

In [ ]:
model = SmallCNN()
print(model)
print(f"\nparámetros entrenables: {count_parameters(model):,}")

In [ ]:
x_train, y_train, x_test, y_test = load_mnist()
train_padded = pad_to_square(x_train, SIZE)
test_padded = pad_to_square(x_test, SIZE)

results, curves, chosen = [], {}, {}

for n in TRAIN_SIZES:
    subset = stratified_subsample(y_train, n, seed=SEED)
    fit_idx, val_idx = train_val_split(y_train, subset, VAL_FRACTION, seed=SEED)

    outcome = train_and_evaluate(
        train_padded[fit_idx], y_train[fit_idx],
        train_padded[val_idx], y_train[val_idx],
        test_padded, y_test,
        seed=SEED,
    )

    curves[n] = outcome.val_curve
    chosen[n] = outcome.best_epoch
    results.append(
        Result(
            model="cnn",
            descriptor="pixels",
            train_size=n,
            seed=SEED,
            error_rate=outcome.test_error,
            best_params={"best_epoch": str(outcome.best_epoch)},
            n_features=SIZE * SIZE,
            fit_seconds=outcome.seconds,
            extra={"n_parameters": outcome.n_parameters, "val_size": len(val_idx)},
        )
    )
    print(f"n={n:5d}  error={outcome.test_error*100:5.2f}%  época={outcome.best_epoch:2d}  ({outcome.seconds:.0f}s)")

save_results(results, "03_cnn_baseline")

In [ ]:
fig = plot_validation_curves(curves, chosen)
save_figure(fig, "fig07_curvas_validacion_cnn")

El ruido de las curvas crece al bajar $n$, tal y como cabe esperar de un
conjunto de validación que encoge con él. Esa es la justificación del suavizado:
no es un truco para mejorar a la CNN sino una defensa contra elegir una época al
azar.

## Contraste con el paper y con el scattering

In [ ]:
reference = pd.read_csv("../results/reference/bruna_mallat_2013_table4.csv", comment="#")
scattering_results = pd.DataFrame(
    __import__("json").loads(open("../results/02_replica_mnist.json", encoding="utf-8").read())["results"]
)

best_scattering = (
    scattering_results[scattering_results.descriptor == "scat2"]
    .groupby("train_size")["error_rate"]
    .min()
    * 100
)

comparison = pd.DataFrame(
    [
        {
            "n": r.train_size,
            "CNN nuestra %": round(100 * r.error_rate, 2),
            "ConvNet paper %": float(reference.loc[reference.train_size == r.train_size, "convnet"].iloc[0]),
            "mejor scattering %": round(best_scattering[r.train_size], 2),
        }
        for r in results
    ]
)
comparison["ventaja del scattering"] = (
    comparison["CNN nuestra %"] - comparison["mejor scattering %"]
).round(2)
comparison

## Cuántos parámetros tiene cada cosa

Aquí conviene ser incómodamente honesto, porque el conteo no dice lo que la
narrativa fácil sugiere.

In [ ]:
from src.classifiers import AffinePCAClassifier
from src.features import PathNormalizer, cached_scattering, flatten, select_orders
from src.scattering import build_scattering, num_learnable_parameters, verify_paths

scattering = build_scattering(J=3, L=8, shape=(SIZE, SIZE), max_order=2)
paths = verify_paths(scattering)
features = cached_scattering(train_padded, scattering, "mnist_train", J=3, L=8, size=SIZE, order=2)

rows = [
    {"componente": "scattering (representación)", "n": "—", "números estimados": num_learnable_parameters(scattering)},
    {"componente": "CNN (entrenables)", "n": "—", "números estimados": count_parameters(SmallCNN())},
]

for n, d in ((300, 5), (1000, 65), (5000, 140)):
    subset = stratified_subsample(y_train, n, seed=SEED)
    descriptor = flatten(PathNormalizer().fit_transform(select_orders(features[subset], paths, 2)))
    classifier = AffinePCAClassifier(n_components=d).fit(descriptor, y_train[subset])
    rows.append({"componente": f"PCA afín (d={d})", "n": n, "números estimados": classifier.n_parameters()})

pd.DataFrame(rows)

## Lectura

**La CNN reproduce al paper.** Nuestros errores quedan cerca de la columna
`ConvNet` de la Tabla 4, así que la red contra la que comparamos no es un
espantapájaros: está entrenada en condiciones.

**El scattering gana con pocos datos.** La ventaja es grande en $n=300$ y se
estrecha al crecer $n$, que es exactamente la forma predicha.

**Pero no gana por tener menos parámetros.** El clasificador PCA afín estima
entre 208.320 y 4.895.520 números, frente a los 75.626 pesos entrenables de la
CNN. Contando todo, el pipeline de scattering maneja *más* números, no menos.
La afirmación "gana porque tiene menos parámetros" es falsa y no debe aparecer
en el documento.

Lo que sí distingue a ambos es **cómo** se obtienen esos números:

- En el scattering, la **representación** está fijada por construcción: 0
  parámetros ajustados a los datos. El sesgo inductivo —invarianza a
  traslaciones, estabilidad a deformaciones— viene de la teoría, no del
  entrenamiento.
- Los números del clasificador PCA son **estadísticos por clase estimados en
  forma cerrada** mediante una SVD, cada clase por separado y sin términos
  cruzados. No son parámetros libres ajustados a un objetivo conjunto.
- Los pesos de la CNN sí son parámetros libres optimizados a la vez sobre un
  objetivo global, y esa es la diferencia que hace que necesiten más datos.

Dicho de otro modo: lo que escasea cuando escasean los datos no son parámetros,
sino **grados de libertad efectivos acoplados**. Formularlo así es más difícil de
defender que contar pesos, pero es lo que los números sostienen.